In [58]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_naver import ChatClovaX

chat = ChatClovaX(model="HCX-005")
file_path = "./example.txt"
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

loader = TextLoader(file_path)

In [59]:
docs = loader.load_and_split(text_splitter=splitter)

docs

[Document(metadata={'source': './example.txt'}, page_content='헌법개정은 국회재적의원 과반수 또는 대통령의 발의로 제안된다. 모든 국민은 법률이 정하는 바에 의하여 국가기관에 문서로 청원할 권리를 가진다. 외국인은 국제법과 조약이 정하는 바에 의하여 그 지위가 보장된다. 사회적 특수계급의 제도는 인정되지 아니하며, 어떠한 형태로도 이를 창설할 수 없다. 국회의원은 국회에서 직무상 행한 발언과 표결에 관하여 국회외에서 책임을 지지 아니한다. 모든 국민은 능력에 따라 균등하게 교육을 받을 권리를 가진다. 제2항의 재판관중 3인은 국회에서 선출하는 자를, 3인은 대법원장이 지명하는 자를 임명한다. 공무원인 근로자는 법률이 정하는 자에 한하여 단결권·단체교섭권 및 단체행동권을 가진다.\n\n헌법재판소는 법관의 자격을 가진 9인의 재판관으로 구성하며, 재판관은 대통령이 임명한다. 국가는 청원에 대하여 심사할 의무를 진다. 국회의원은 그 지위를 남용하여 국가·공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리·이익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다. 이 헌법은 1988년 2월 25일부터 시행한다. 다만, 이 헌법을 시행하기 위하여 필요한 법률의 제정·개정과 이 헌법에 의한 대통령 및 국회의원의 선거 기타 이 헌법시행에 관한 준비는 이 헌법시행 전에 할 수 있다. 국민경제의 발전을 위한 중요정책의 수립에 관하여 대통령의 자문에 응하기 위하여 국민경제자문회의를 둘 수 있다.'),
 Document(metadata={'source': './example.txt'}, page_content='국회는 법률에 저촉되지 아니하는 범위안에서 의사와 내부규율에 관한 규칙을 제정할 수 있다. 대통령은 국민의 보통·평등·직접·비밀선거에 의하여 선출한다. 누구든지 병역의무의 이행으로 인하여 불이익한 처우를 받지 아니한다. 연소자의 근로는 특별한 보호를 받는다. 신체장애자 및 질병·노령 기타의 사유로 

In [60]:
initial_prompt = PromptTemplate.from_template(
    """
        다음 문서를 요약해줘.
        ----------
        {context}
        ----------
    """
)
initial_chain = initial_prompt | chat | StrOutputParser()
summary = initial_chain.invoke({"context": docs[0].page_content})

summary

'이 문서는 대한민국의 헌법 일부 내용을 다루고 있으며 다음과 같은 주요 내용이 포함되어 있습니다.\n\n1. 헌법 개정은 국회의원 과반수의 발의나 대통령 발의를 통해 제안됩니다.\n2. 모든 국민은 국가 기관에 문서로 청원할 권리가 있고, 외국인은 국제법과 조약에 따라 지위가 보장됩니다.\n3. 사회적 특수 계급 제도를 인정하지 않으며, 이를 창설할 수 없습니다.\n4. 국회의원은 국회 외부에서 직무와 관련된 발언과 투표에 대한 책임을 지지 않습니다.\n5. 모든 국민은 능력에 따른 교육 권리를 가지며, 헌법재판소는 9명의 재판관으로 구성되어 대통령이 임명합니다.\n6. 국회의원은 지위 남용으로 국가, 공공단체, 기업과의 계약으로 이익을 얻거나 타인이 얻게끔 알선할 수 없습니다.\n7. 이 헌법은 1988년 2월 25일부터 시행되며, 필요한 법률 제정 및 개정 등은 시행 전에도 가능합니다.\n8. 국민 경제 자문 회의는 대통령 자문을 위해 설치될 수 있습니다.\n\n이러한 내용들은 국민의 기본적인 권리와 의무, 국회의 기능과 책임, 헌법재판소의 조직과 역할 등에 대해 규정하고 있습니다.'

In [62]:
summary_prompt = PromptTemplate.from_template(
    """
        기존의 요약본과 새로운 문서를 제공할거야.
        주어진 새로운 문서로 기존의 요약본을 조정해서 최종 요약본을 작성해줘.
        추가하거나 수정할 내용이 없다면 기존의 요약본을 그대로 반환해줘.
        주어진 문서만을 참고해야해. 임의의 내용을 반영해서는 안돼.
        또한, 부가적인 설명 없이 그저 요약본 그 자체만 반환해줘.

        기존의 요약본:
        {previous_summary}
        새로운 문서:
        ----------
        {context}
        ----------
    """
)
summary_chain = summary_prompt | chat | StrOutputParser()
for doc in docs[1:3]:
    print("===")
    print(summary)
    summary = summary_chain.invoke(
        {"previous_summary": summary, "context": doc.page_content}
    )

print("final_summary")
print(summary)

===
기존의 요약본과 새로운 문서를 참고하여 다음과 같이 조정하였습니다.

1. 대한민국은 민주공화국으로, 국민의 기본적인 권리와 의무, 국회의 기능과 책임, 정부의 역할, 경제 관련 정책, 언론의 자유와 책임, 정당 제도 등에 대한 내용들이 헌법에 명시되어 있습니다.
2. 국가는 전직 대통령의 신분과 예우를 법률로 정하며, 건전한 소비 행위를 계도하고 생산품의 품질 향상을 촉구하기 위한 소비자 보호 운동을 법률이 정하는 바에 의해 보장합니다.
3. 국가는 국민의 자유와 권리를 보호하며, 헌법에서 명시되지 않은 이유로 경시되지 않도록 합니다.
4. 국회는 국가의 예산안을 심의 확정하며, 모든 국민의 재산권을 보장하되 그 내용과 한계는 법률로 정합니다.
5. 국무총리는 국회의 동의를 얻어 대통령이 임명하며, 선거에서는 최고 득표자가 2인 이상일 때 재적의원 과반수가 출석한 공개회의에서 다수 표를 얻은 자가 당선됩니다.
6. 정부는 정당 설립의 자유를 보장하며, 복수 정당제를 지원합니다. 또한, 대통령은 국무 회의의 의장으로서 국정 운영에 참여하며, 국무총리는 부의장입니다. 대법원장 제외한 법관의 임명과 국정 감사 및 조사 등에 관한 절차 등은 모두 법률로 정해집니다.
7. 국군은 정치적 중립성을 지키면서 국가의 안전보장과 국토 방위라는 중요한 임무를 수행하며, 국가유공자, 상이군경, 전몰 군경 가족들은 법률에 따라 우선적으로 취업 기회를 제공받습니다.

이와 같이 대한민국의 정치 체제와 국민의 권리 보호를 중심으로 한 여러 원칙과 규정이 포함되어 있음을 알 수 있습니다.
===
대한민국의 정치 체제와 국민의 권리 보호를 중심으로 한 여러 원칙과 규정은 아래와 같습니다.

1. 국회는 법률에 저촉되지 않는 범위 안에서 의사 결정과 내부 규정에 관한 규칙을 제정할 수 있으며, 대통령은 국민의 직접 선거로 뽑힙니다.
2. 국민 모두가 병역 의무를 다하더라도 어떠한 불이익도 받지 않으며, 어린이나 청소년 근로자들은 특별히 보호받습니다.
3. 생활 능력이 부족한 자들(